# 04 - Deep Dive: Error Analysis

This notebook analyses the prediction errors of our best model to understand
failure modes, identify systematic biases, and discover opportunities for improvement.

**Sections:**
1. Load predictions from best model
2. Confusion matrix analysis
3. Hardest matches to predict (by match characteristics)
4. Error rate by round number (early vs late season)
5. Error rate by team
6. Upset analysis (heavy favourites losing)
7. Confidence vs accuracy
8. Lessons learned and potential improvements

In [ ]:
# ============================================================
# Cell 1: Imports and load best model predictions
# ============================================================
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from sklearn.metrics import confusion_matrix, classification_report

from config.settings import FEATURES_DIR, PROCESSED_DIR, OUTPUTS_DIR
from evaluation.metrics import compute_all_metrics

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------------
# Load predictions
# ------------------------------------------------------------------
preds_path = OUTPUTS_DIR / "best_model_predictions.parquet"
if preds_path.exists():
    preds = pd.read_parquet(preds_path)
    print(f"Loaded predictions: {len(preds):,} rows")
else:
    # Fallback: try CSV
    preds_csv = OUTPUTS_DIR / "best_model_predictions.csv"
    if preds_csv.exists():
        preds = pd.read_csv(preds_csv)
        print(f"Loaded predictions from CSV: {len(preds):,} rows")
    else:
        print(f"WARNING: No predictions file found at {preds_path}")
        print("Please run notebook 03_model_comparison first to generate predictions.")
        preds = pd.DataFrame()

# Load the feature data for match context
features_path = FEATURES_DIR / "features.parquet"
alt_paths = list(FEATURES_DIR.glob("*.parquet")) + list(FEATURES_DIR.glob("*.csv"))
if features_path.exists():
    features = pd.read_parquet(features_path)
elif alt_paths:
    fpath = alt_paths[0]
    features = pd.read_parquet(fpath) if fpath.suffix == ".parquet" else pd.read_csv(fpath)
else:
    features = pd.DataFrame()

# Build target column in features if needed
if not features.empty and "home_win" not in features.columns:
    if "home_score" in features.columns and "away_score" in features.columns:
        features["home_win"] = (features["home_score"] > features["away_score"]).astype(int)

# Add match context to predictions if possible
year_col = "season" if "season" in features.columns else "year"
if not preds.empty and not features.empty and "year" in preds.columns:
    # Try to match predictions back to features using year and index
    # The backtester outputs year, y_true, y_pred, y_prob
    test_years = sorted(preds["year"].unique())
    print(f"Test years in predictions: {test_years}")

    # Reconstruct match context from features
    context_cols = ["home_team", "away_team", "venue", "round", "date"]
    available_ctx = [c for c in context_cols if c in features.columns]

    if available_ctx and year_col in features.columns:
        test_features = features[features[year_col].isin(test_years)].reset_index(drop=True)
        if len(test_features) == len(preds):
            for col in available_ctx:
                preds[col] = test_features[col].values
            print(f"Added context columns: {available_ctx}")
        else:
            print(f"Length mismatch: predictions={len(preds)}, features={len(test_features)}")
            print("Context columns not added.")

# Add derived columns
if not preds.empty:
    preds["correct"] = (preds["y_true"] == preds["y_pred"]).astype(int)
    preds["confidence"] = preds["y_prob"].apply(lambda p: max(p, 1 - p))
    preds["predicted_home"] = (preds["y_pred"] == 1).astype(int)
    preds["actual_home"] = (preds["y_true"] == 1).astype(int)

    overall = compute_all_metrics(preds["y_true"], preds["y_pred"], preds["y_prob"])
    print(f"\nOverall performance:")
    for k, v in overall.items():
        print(f"  {k:15s}: {v:.4f}")
    print(f"  Total matches:   {len(preds):,}")
    print(f"  Correct:         {preds['correct'].sum():,} ({preds['correct'].mean():.1%})")
    print(f"  Errors:          {(1 - preds['correct']).sum():.0f} ({1 - preds['correct'].mean():.1%})")

In [ ]:
# ============================================================
# Cell 2: Confusion matrix analysis
# ============================================================

if not preds.empty:
    y_true = preds["y_true"].values
    y_pred = preds["y_pred"].values

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Panel 1: Raw counts
    ax = axes[0]
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Away Win", "Home Win"],
        yticklabels=["Away Win", "Home Win"],
        ax=ax,
        annot_kws={"size": 16},
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix (Counts)", fontsize=12, fontweight="bold")

    # Panel 2: Normalized (row-wise)
    ax = axes[1]
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt=".1%", cmap="RdYlGn",
        xticklabels=["Away Win", "Home Win"],
        yticklabels=["Away Win", "Home Win"],
        ax=ax,
        vmin=0, vmax=1,
        annot_kws={"size": 14},
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix (Normalized by Row)", fontsize=12, fontweight="bold")

    plt.tight_layout()
    plt.show()

    # Print classification report
    print("\nClassification Report:")
    print("=" * 60)
    print(classification_report(
        y_true, y_pred,
        target_names=["Away Win", "Home Win"],
        digits=4,
    ))

    # Error type breakdown
    tn, fp, fn, tp = cm.ravel()
    print("\nError Breakdown:")
    print("-" * 40)
    print(f"  True Negatives  (correct away win):  {tn:4d}")
    print(f"  True Positives  (correct home win):  {tp:4d}")
    print(f"  False Positives (predicted home, was away): {fp:4d}  ({fp/(fp+tn)*100:.1f}% of actual away wins)")
    print(f"  False Negatives (predicted away, was home): {fn:4d}  ({fn/(fn+tp)*100:.1f}% of actual home wins)")
    print(f"\n  Home win bias: model predicts home {preds['predicted_home'].mean()*100:.1f}% of the time")
else:
    print("No predictions available for analysis.")

In [ ]:
# ============================================================
# Cell 3: Which matches are hardest to predict?
# ============================================================

if not preds.empty:
    errors = preds[preds["correct"] == 0].copy()
    correct = preds[preds["correct"] == 1].copy()

    print(f"Total errors: {len(errors):,} out of {len(preds):,} ({len(errors)/len(preds)*100:.1f}%)")

    # Confidence distribution: errors vs correct predictions
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1: Confidence distribution split
    ax = axes[0]
    bins = np.arange(0.50, 1.01, 0.02)
    ax.hist(correct["confidence"], bins=bins, alpha=0.6, label="Correct",
            color="#2ca02c", density=True, edgecolor="white")
    ax.hist(errors["confidence"], bins=bins, alpha=0.6, label="Errors",
            color="#d62728", density=True, edgecolor="white")
    ax.set_xlabel("Model Confidence")
    ax.set_ylabel("Density")
    ax.set_title("Confidence Distribution: Correct vs Errors", fontweight="bold")
    ax.legend()

    # Panel 2: Probability distribution for errors
    ax = axes[1]
    ax.hist(errors["y_prob"], bins=20, color="#d62728", alpha=0.7, edgecolor="white")
    ax.axvline(0.5, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Predicted P(home_win)")
    ax.set_ylabel("Count")
    ax.set_title("Probability Distribution of Errors", fontweight="bold")

    # Panel 3: Error rate by confidence bucket
    ax = axes[2]
    preds["conf_bucket"] = pd.cut(preds["confidence"], bins=np.arange(0.5, 1.05, 0.05))
    bucket_stats = (
        preds.groupby("conf_bucket", observed=False)
        .agg(n=("correct", "count"), error_rate=("correct", lambda x: 1 - x.mean()))
    )
    bucket_stats = bucket_stats[bucket_stats["n"] > 0]

    x_labels = [str(b) for b in bucket_stats.index]
    bars = ax.bar(range(len(bucket_stats)), bucket_stats["error_rate"],
                  color="#ff7f0e", edgecolor="white")
    ax.set_xticks(range(len(bucket_stats)))
    ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Confidence Bucket")
    ax.set_ylabel("Error Rate")
    ax.set_title("Error Rate by Confidence Bucket", fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

    # Annotate with sample counts
    for bar, n in zip(bars, bucket_stats["n"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"n={n:.0f}", ha="center", fontsize=7)

    plt.tight_layout()
    plt.show()

    # Most confident errors
    print("\nMost confident ERRORS (model was very sure but wrong):")
    print("-" * 60)
    top_errors = errors.nlargest(10, "confidence")
    display_cols = [c for c in ["year", "round", "home_team", "away_team", "y_prob", "confidence", "y_true", "y_pred"]
                    if c in top_errors.columns]
    display(top_errors[display_cols].reset_index(drop=True))
else:
    print("No predictions available.")

In [ ]:
# ============================================================
# Cell 4: Error rate by round number
# ============================================================

if not preds.empty and "round" in preds.columns:
    # Convert round to numeric where possible
    preds["round_num"] = pd.to_numeric(preds["round"], errors="coerce")

    round_stats = (
        preds.dropna(subset=["round_num"])
        .groupby("round_num")
        .agg(
            n_matches=("correct", "count"),
            accuracy=("correct", "mean"),
            avg_confidence=("confidence", "mean"),
        )
        .sort_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Panel 1: Accuracy by round
    ax = axes[0]
    ax.plot(round_stats.index, round_stats["accuracy"], marker="o",
            markersize=6, linewidth=1.5, color="#1f77b4", label="Accuracy")

    # Rolling average
    if len(round_stats) >= 5:
        rolling = round_stats["accuracy"].rolling(window=5, min_periods=1).mean()
        ax.plot(round_stats.index, rolling, linewidth=2.5, linestyle="--",
                color="#ff7f0e", label="5-round rolling avg")

    ax.axhline(preds["correct"].mean(), color="green", linestyle="-.",
               linewidth=1, label=f"Overall: {preds['correct'].mean():.1%}")
    ax.axhline(0.5, color="red", linestyle=":", linewidth=0.8, alpha=0.5)
    ax.set_xlabel("Round")
    ax.set_ylabel("Accuracy")
    ax.set_title("Prediction Accuracy by Round", fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.legend(loc="lower right", fontsize=9)
    ax.set_ylim(0.3, 0.85)

    # Panel 2: Error rate for early, mid, late season
    ax = axes[1]
    preds_with_round = preds.dropna(subset=["round_num"]).copy()
    preds_with_round["season_phase"] = pd.cut(
        preds_with_round["round_num"],
        bins=[0, 8, 18, 27, 100],
        labels=["Early\n(Rd 1-8)", "Mid\n(Rd 9-18)", "Late\n(Rd 19-27)", "Finals"],
    )

    phase_stats = (
        preds_with_round.groupby("season_phase", observed=False)
        .agg(
            n=("correct", "count"),
            accuracy=("correct", "mean"),
            error_rate=("correct", lambda x: 1 - x.mean()),
        )
    )
    phase_stats = phase_stats[phase_stats["n"] > 0]

    colors = ["#ff7f0e" if er > 0.35 else "#1f77b4" for er in phase_stats["error_rate"]]
    bars = ax.bar(phase_stats.index.astype(str), phase_stats["accuracy"],
                  color=colors, edgecolor="white")
    ax.axhline(preds["correct"].mean(), color="green", linestyle="--",
               label=f"Overall: {preds['correct'].mean():.1%}")

    for bar, (_, row) in zip(bars, phase_stats.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{row['accuracy']:.1%}\n(n={row['n']:.0f})", ha="center", fontsize=9)

    ax.set_ylabel("Accuracy")
    ax.set_title("Accuracy by Season Phase", fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.set_ylim(0, 0.85)
    ax.legend()

    plt.tight_layout()
    plt.show()

    print("\nAccuracy by Season Phase:")
    print("-" * 50)
    for phase, row in phase_stats.iterrows():
        print(f"  {str(phase):20s}: {row['accuracy']:.1%} (n={row['n']:.0f})")
else:
    print("Round information not available in predictions.")
    if not preds.empty and "year" in preds.columns:
        # Fallback: accuracy by year
        year_acc = preds.groupby("year")["correct"].mean()
        print("\nAccuracy by Year:")
        for yr, acc in year_acc.items():
            print(f"  {yr}: {acc:.1%}")

In [ ]:
# ============================================================
# Cell 5: Error rate by team
# ============================================================

if not preds.empty and "home_team" in preds.columns and "away_team" in preds.columns:
    # Build per-team error stats
    team_records = []

    for _, row in preds.iterrows():
        # Home team
        team_records.append({
            "team": row["home_team"],
            "role": "home",
            "correct": row["correct"],
            "confidence": row["confidence"],
        })
        # Away team
        team_records.append({
            "team": row["away_team"],
            "role": "away",
            "correct": row["correct"],
            "confidence": row["confidence"],
        })

    team_df = pd.DataFrame(team_records)

    team_error_stats = (
        team_df.groupby("team")
        .agg(
            n_matches=("correct", "count"),
            accuracy=("correct", "mean"),
            avg_confidence=("confidence", "mean"),
        )
        .sort_values("accuracy", ascending=True)
    )

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # Panel 1: Accuracy by team (all matches involving team)
    ax = axes[0]
    colors = plt.cm.RdYlGn(np.linspace(0.15, 0.9, len(team_error_stats)))
    ax.barh(team_error_stats.index, team_error_stats["accuracy"],
            color=colors, edgecolor="white")
    ax.axvline(preds["correct"].mean(), color="red", linestyle="--",
               linewidth=1, label=f"Overall: {preds['correct'].mean():.1%}")
    ax.set_xlabel("Prediction Accuracy (for matches involving team)")
    ax.set_title("Which teams are hardest to predict?", fontsize=12, fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.legend()

    for i, (team, row) in enumerate(team_error_stats.iterrows()):
        ax.text(row["accuracy"] + 0.005, i,
                f"{row['accuracy']:.0%} (n={row['n_matches']:.0f})",
                va="center", fontsize=7)

    # Panel 2: Accuracy split by home/away
    ax = axes[1]
    role_team_stats = (
        team_df.groupby(["team", "role"])["correct"]
        .mean()
        .unstack(fill_value=0)
    )
    role_team_stats = role_team_stats.sort_values("home", ascending=True)

    y_pos = np.arange(len(role_team_stats))
    width = 0.35

    if "home" in role_team_stats.columns:
        ax.barh(y_pos + width/2, role_team_stats["home"], width,
                label="When Home", color="#1f77b4", alpha=0.8)
    if "away" in role_team_stats.columns:
        ax.barh(y_pos - width/2, role_team_stats["away"], width,
                label="When Away", color="#ff7f0e", alpha=0.8)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(role_team_stats.index, fontsize=8)
    ax.axvline(0.5, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Prediction Accuracy")
    ax.set_title("Accuracy Split: Home vs Away", fontsize=12, fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.legend(loc="lower right")

    plt.tight_layout()
    plt.show()

    # Print the most and least predictable teams
    print("\nMost PREDICTABLE teams (highest accuracy):")
    for team, row in team_error_stats.tail(5).iloc[::-1].iterrows():
        print(f"  {team:35s}: {row['accuracy']:.1%} (n={row['n_matches']:.0f})")

    print("\nMost UNPREDICTABLE teams (lowest accuracy):")
    for team, row in team_error_stats.head(5).iterrows():
        print(f"  {team:35s}: {row['accuracy']:.1%} (n={row['n_matches']:.0f})")
else:
    print("Team names not available in predictions for team-level analysis.")

In [ ]:
# ============================================================
# Cell 6: Upset analysis - when heavy favourites lose
# ============================================================

if not preds.empty:
    # Define "heavy favourite" as confidence >= 0.65
    UPSET_THRESHOLD = 0.65

    # Find matches where the model was very confident
    high_conf = preds[preds["confidence"] >= UPSET_THRESHOLD].copy()
    upsets = high_conf[high_conf["correct"] == 0]

    print(f"Upset Analysis (confidence >= {UPSET_THRESHOLD:.0%})")
    print("=" * 60)
    print(f"  High-confidence predictions: {len(high_conf):,}")
    print(f"  Upsets (wrong): {len(upsets):,} ({len(upsets)/len(high_conf)*100:.1f}%)")
    print(f"  Correct: {len(high_conf) - len(upsets):,} ({(len(high_conf) - len(upsets))/len(high_conf)*100:.1f}%)")

    # Confidence-based upset analysis
    conf_buckets = np.arange(0.50, 1.00, 0.05)
    upset_rates = []

    for lo in conf_buckets:
        hi = lo + 0.05
        bucket = preds[(preds["confidence"] >= lo) & (preds["confidence"] < hi)]
        if len(bucket) > 0:
            upset_rates.append({
                "confidence": f"{lo:.0%}-{hi:.0%}",
                "conf_mid": (lo + hi) / 2,
                "n_matches": len(bucket),
                "n_upsets": (1 - bucket["correct"]).sum(),
                "upset_rate": 1 - bucket["correct"].mean(),
                "expected_error": 1 - (lo + hi) / 2,
            })

    upset_df = pd.DataFrame(upset_rates)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Panel 1: Actual vs expected error rate by confidence
    ax = axes[0]
    ax.bar(range(len(upset_df)), upset_df["upset_rate"],
           color="#d62728", alpha=0.7, label="Actual Error Rate", edgecolor="white")
    ax.plot(range(len(upset_df)), upset_df["expected_error"],
            "ko--", linewidth=1.5, markersize=6, label="Expected Error Rate")
    ax.set_xticks(range(len(upset_df)))
    ax.set_xticklabels(upset_df["confidence"], rotation=45, ha="right", fontsize=8)
    ax.set_xlabel("Confidence Bucket")
    ax.set_ylabel("Error Rate")
    ax.set_title("Actual vs Expected Error Rate", fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax.legend()

    # Panel 2: Upset count distribution by year
    ax = axes[1]
    if "year" in preds.columns:
        # Upsets per year
        high_conf_by_year = (
            high_conf.groupby("year")
            .agg(
                n_high_conf=("correct", "count"),
                n_upsets=("correct", lambda x: (1 - x).sum()),
                upset_rate=("correct", lambda x: 1 - x.mean()),
            )
        )

        x = range(len(high_conf_by_year))
        ax.bar(x, high_conf_by_year["n_upsets"], color="#d62728",
               alpha=0.7, label="Upsets", edgecolor="white")
        ax.set_xticks(x)
        ax.set_xticklabels(high_conf_by_year.index.astype(int), rotation=45)
        ax.set_xlabel("Season")
        ax.set_ylabel("Number of Upsets")
        ax.set_title(f"High-Confidence Upsets by Season (conf >= {UPSET_THRESHOLD:.0%})",
                     fontsize=12, fontweight="bold")

        # Add upset rate as text
        for i, (_, row) in enumerate(high_conf_by_year.iterrows()):
            ax.text(i, row["n_upsets"] + 0.3,
                    f"{row['upset_rate']:.0%}\n(n={row['n_high_conf']:.0f})",
                    ha="center", fontsize=7)
    else:
        ax.text(0.5, 0.5, "Year data not available", transform=ax.transAxes, ha="center")

    plt.tight_layout()
    plt.show()

    # Show the biggest upsets
    if len(upsets) > 0:
        print(f"\nTop 10 Biggest Upsets (highest confidence errors):")
        print("-" * 60)
        display_cols = [c for c in ["year", "round", "home_team", "away_team", "confidence", "y_prob", "y_true"]
                        if c in upsets.columns]
        display(upsets.nlargest(10, "confidence")[display_cols].reset_index(drop=True))
else:
    print("No predictions available for upset analysis.")

In [ ]:
# ============================================================
# Cell 7: Confidence vs accuracy plot
# ============================================================

if not preds.empty:
    # Create fine-grained confidence bins for calibration analysis
    n_bins = 20
    preds["prob_bin"] = pd.cut(preds["y_prob"], bins=np.linspace(0, 1, n_bins + 1))

    bin_stats = (
        preds.groupby("prob_bin", observed=False)
        .agg(
            n=("y_true", "count"),
            mean_prob=("y_prob", "mean"),
            actual_rate=("y_true", "mean"),
        )
    )
    bin_stats = bin_stats[bin_stats["n"] > 0]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Panel 1: Reliability diagram (confidence vs accuracy)
    ax = axes[0]
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration")
    ax.scatter(
        bin_stats["mean_prob"], bin_stats["actual_rate"],
        s=bin_stats["n"] * 2,  # Size proportional to sample count
        c="#1f77b4", alpha=0.8, edgecolors="black", linewidth=0.5,
        label="Model (size = n samples)",
    )
    ax.plot(bin_stats["mean_prob"], bin_stats["actual_rate"],
            color="#1f77b4", alpha=0.5, linewidth=1)

    # Annotate calibration quality
    from evaluation.metrics import compute_ece
    ece = compute_ece(preds["y_true"], preds["y_prob"])
    ax.text(0.05, 0.92, f"ECE = {ece:.4f}", transform=ax.transAxes,
            fontsize=11, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.7))

    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Observed Frequency (Actual Home Win Rate)")
    ax.set_title("Confidence vs Accuracy (Reliability Diagram)", fontsize=12, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)

    # Panel 2: Accuracy curve as a function of confidence threshold
    ax = axes[1]
    thresholds = np.arange(0.50, 0.95, 0.01)
    accs = []
    coverages = []

    for thresh in thresholds:
        mask = preds["confidence"] >= thresh
        n_above = mask.sum()
        if n_above > 0:
            accs.append(preds.loc[mask, "correct"].mean())
            coverages.append(n_above / len(preds))
        else:
            accs.append(np.nan)
            coverages.append(0)

    ax2 = ax.twinx()
    ax.plot(thresholds, accs, color="#1f77b4", linewidth=2, label="Accuracy")
    ax2.plot(thresholds, coverages, color="#ff7f0e", linewidth=2, linestyle="--",
             label="Coverage")

    ax.set_xlabel("Minimum Confidence Threshold")
    ax.set_ylabel("Accuracy", color="#1f77b4")
    ax2.set_ylabel("Coverage (fraction of matches)", color="#ff7f0e")
    ax.set_title("Accuracy vs Coverage Trade-off", fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))
    ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0))

    # Combined legend
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="center right")

    plt.tight_layout()
    plt.show()

    # Print key thresholds
    print("\nAccuracy at Key Confidence Thresholds:")
    print("-" * 50)
    for thresh in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
        mask = preds["confidence"] >= thresh
        n = mask.sum()
        if n > 0:
            acc = preds.loc[mask, "correct"].mean()
            print(f"  Conf >= {thresh:.0%}: accuracy={acc:.1%}, coverage={n/len(preds):.1%} (n={n:,})")
else:
    print("No predictions available.")

In [ ]:
# ============================================================
# Cell 8: Lessons learned and potential improvements
# ============================================================

if not preds.empty:
    print("=" * 70)
    print("ERROR ANALYSIS SUMMARY & IMPROVEMENT IDEAS")
    print("=" * 70)

    # Calculate key summary stats
    overall_acc = preds["correct"].mean()
    total_errors = (1 - preds["correct"]).sum()

    # Errors by confidence level
    low_conf = preds[preds["confidence"] < 0.55]
    med_conf = preds[(preds["confidence"] >= 0.55) & (preds["confidence"] < 0.65)]
    high_conf_errs = preds[(preds["confidence"] >= 0.65) & (preds["correct"] == 0)]

    print(f"\n1. OVERALL PERFORMANCE")
    print(f"   - Accuracy: {overall_acc:.1%}")
    print(f"   - Total errors: {total_errors:.0f}")

    print(f"\n2. ERROR DISTRIBUTION BY CONFIDENCE")
    print(f"   - Low confidence (<55%): {len(low_conf):,} matches, "
          f"accuracy={low_conf['correct'].mean():.1%}")
    print(f"   - Medium confidence (55-65%): {len(med_conf):,} matches, "
          f"accuracy={med_conf['correct'].mean():.1%}")
    print(f"   - High confidence (>=65%) errors: {len(high_conf_errs):,}")

    # Check for home bias
    home_pred_rate = preds["predicted_home"].mean()
    actual_home_rate = preds["actual_home"].mean()

    print(f"\n3. HOME BIAS ANALYSIS")
    print(f"   - Model predicts home win: {home_pred_rate:.1%}")
    print(f"   - Actual home win rate: {actual_home_rate:.1%}")
    print(f"   - Bias: {(home_pred_rate - actual_home_rate) * 100:+.1f} pp")

    # Calibration quality
    ece = compute_ece(preds["y_true"], preds["y_prob"])
    print(f"\n4. CALIBRATION")
    print(f"   - ECE: {ece:.4f}")
    quality = "Good" if ece < 0.03 else "Fair" if ece < 0.06 else "Needs improvement"
    print(f"   - Quality: {quality}")

    print(f"\n5. POTENTIAL IMPROVEMENTS")
    print("   a) Feature Engineering:")
    print("      - Weather and pitch conditions (currently not included)")
    print("      - Player-level injury/suspension data")
    print("      - Travel fatigue (especially for NZ Warriors)")
    print("      - Recent coaching changes")
    print("   b) Modelling:")
    if ece > 0.03:
        print("      - Apply probability calibration (Platt scaling or isotonic)")
    print("      - Try ensemble methods (stacking top models)")
    print("      - Hyperparameter tuning via Optuna")
    print("      - Explore time-weighted training (downweight older seasons)")
    print("   c) Strategy:")
    print("      - Focus on high-confidence predictions for betting")
    print("      - Consider abstaining on low-confidence matches")

    # Quick win analysis: how much would abstaining help?
    print(f"\n6. ABSTENTION ANALYSIS")
    for min_conf in [0.55, 0.60, 0.65, 0.70]:
        mask = preds["confidence"] >= min_conf
        if mask.sum() > 0:
            acc = preds.loc[mask, "correct"].mean()
            cov = mask.mean()
            print(f"   If only betting on conf >= {min_conf:.0%}: "
                  f"accuracy={acc:.1%}, coverage={cov:.0%} of matches")
else:
    print("No predictions data available for summary analysis.")